# 实战案例：基于注意力的图像描述

## 文件目录与提交规范

**本作业要求使用PyTorch实现。请补全标有 `# TODO` 的代码。**

所有路径均相对于本作业文件夹。原始数据固定放在 `data/flickr8k/`，其中包含 `dataset_flickr8k.json` 和 `images/`。生成的 `vocab.json`、`train_data.json`、`val_data.json`、`test_data.json` 也保存在 `data/flickr8k/`。

模型统一保存在 `work/` 文件夹中，模型文件名自行确定；代码中的文件名仅为示例，可自行修改。运行时自动创建 `work/`。提交完成的 Notebook 与 `work/` 文件夹，不提交 `data/` 数据集。


## 图像描述技术简介

图像描述的关键是生成自然语言描述图像中可以用语言表述的部分。传统的图像描述技术首先通过分析视觉内容来预测给定图像最可能包含的语义信息，并显式的转化为语言标签（通常为单词、短语或其他结构化描述），再基于这些标签生成自然语言描述句子。这类方法均使用以下的管道式结构实现图像描述任务： 

（1）使用计算机视觉技术来对场景进行分类，检测图像中存在的对象，预测它们的属性以及它们之间的关系，识别发生的动作，将它们映射为一些基本的自然语言描述单元，例如单词、短语或其他结构化描述。 

（2）通过自然语言生成技术（例如，模板，n-gram，语法规则等）将这些单词或者短语进行组合，生成自然语言描述句子。

这种管道式方法虽然充分利用了两个领域的现有技术，设计了一套简单可控的解决方案，然而，也存在若干问题：其一，分阶段的方式限制了两个模态数据间的信息交互；其二，这种方法高度依赖于预先定义的场景、对象、属性和动作的封闭语义类集；其三，这种分阶段的模型存在误差累积问题，前面任务的误差在后面阶段会放大；其四，训练误差不能前向传递。

当前主流的图像描述技术大多采用基于编解码框架的方法直接学习图像到文本描述的映射，其核心思想是建模一个以图像为条件的语言模型，计算视觉模式与文本模式的共现概率；其技术基础是深层神经网络对图文两种不同模态数据的通用表示学习能力，可以形成一个端到端的编码解码模型结构。此类方法中所使用的模型可以被端到端地训练，且不需要显示地定义图像和文本之间的桥梁（状态表示），可以有效避免前述管道式方法的问题。

不同的图像描述编解码模型的区别在于其图像编码器和文本解码器所使用的结构的不同。下表列举了深度学习时代常见的图像描述编解码器组合。

| 图像编码器 | 文本解码器 | 
| :----: | :----: | 
| 整体表示 | RNN | 
| 局部表示 | RNN+注意力 | 
| 局部表示+自注意力 | RNN+注意力 | 
| 局部表示+图网络 | RNN+注意力 | 
| 局部表示+Transformer编码器 | Transformer解码器 | 
| 视觉Transformer | Transformer解码模块 | 

接下来，我们将介绍一个图像编码器为CNN网格表示提取器、文本解码器为RNN+注意力的图像描述方法的具体实现。我们的实现大体上是在复现ARCTIC模型，但是在细节上有一些改变，下面的实现过程会对这些改变做具体说明。此外，[链接](https://github.com/sgrvinod/a-PyTorch-Tutorial-to-Image-Captioning)给出了一个更接近原始ARCTIC模型的代码库，非常推荐大家阅读。本节的部分代码也是受到该代码库的启发。

下面，按照读取数据、定义模型、定义损失函数、选择优化方法、选择评估指标和训练模型的次序，来描述该实战案例。

## 1. 环境初始化与数据定位

本单元导入 PyTorch、NumPy、PIL 等依赖，固定随机种子并定位作业目录。默认以当前工作目录作为根目录，也可通过代码中的作业目录环境变量指定根目录。

请提前将图片和划分文件整理到 `data/flickr8k/`：`images/` 存放图片，`dataset_flickr8k.json` 保存图片、caption 和官方 split。程序会检查这两个输入；缺失时会提示错误。本单元不执行解压操作。

默认从官方 train/val/test 各取 128/32/32 张图片，每图5条描述，原始文本最多保留20个 token。运行后应先核对打印的数据来源。

In [ ]:
from pathlib import Path
import json
import math
import os
import random
from collections import Counter, defaultdict
from types import SimpleNamespace

import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as tv_models
import torchvision.transforms as tv_transforms

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
configured_root = os.environ.get("HOMEWORK2_DIR")
NOTEBOOK_DIR = (
    Path(configured_root).expanduser().resolve()
    if configured_root else Path.cwd().resolve()
)
SOURCE_DIR = NOTEBOOK_DIR / "data" / "flickr8k"
if not (SOURCE_DIR / "dataset_flickr8k.json").is_file() or not (SOURCE_DIR / "images").is_dir():
    raise FileNotFoundError(
        f"请将数据放在 {SOURCE_DIR}，该目录须包含 dataset_flickr8k.json 和 images/。"
    )
OUTPUT_DIR = SOURCE_DIR
SPLIT_LIMITS = {"train": 128, "val": 32, "test": 32}
CAPTIONS_PER_IMAGE = 5
MAX_LEN = 20
print(f"使用设备: {device}")
print(f"数据来源: {SOURCE_DIR}")

## 2. 整理数据集与构建词表

原始 JSON 中的 `split` 定义了训练、验证和测试划分。先分别取各 split 的前 N 张图片，再将自然语言描述编码为整数序列；图片只记录路径，读取样本时再加载图像。

词表只统计所选训练图片的 caption，保留出现至少2次的词，并加入 `<pad>`、`<unk>`、`<start>`、`<end>`。验证和测试中的未登录词映射为 `<unk>`，不能参与词表构建。文本截断后在首尾加入开始和结束标记。

输出 `vocab.json`、`train_data.json`、`val_data.json`、`test_data.json`，均保存在 `data/flickr8k/`。默认生成640条训练描述和各160条验证、测试描述。

**任务与提示：** 对照 TODO 1 完成固定数量的 split 选择，理解图片列表与 caption 列表的对应关系，并核对生成数量。

In [ ]:
def prepare_cpu_quick_data(source_dir, output_dir, split_limits, max_len=20):
    """按 Karpathy split 取固定前 N 张图，生成可复现的小规模数据。"""
    source_dir = Path(source_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    raw = json.loads((source_dir / "dataset_flickr8k.json").read_text())
    by_split = defaultdict(list)
    for item in raw["images"]:
        if item.get("split") in split_limits:
            by_split[item["split"]].append(item)

    selected_by_split = {}
    for split, limit in split_limits.items():
        # STUDENT_TODO 1：依据 split 选择固定数量图片（提示：对 by_split[split] 切片到 limit）
        raise NotImplementedError("请完成 split 选择")
        if len(selected) != limit:
            raise ValueError(f"{split} 仅找到 {len(selected)} 张图，需要 {limit} 张")
        selected_by_split[split] = selected

    vocab_counter = Counter()
    for item in selected_by_split["train"]:
        for sentence in item["sentences"][:CAPTIONS_PER_IMAGE]:
            vocab_counter.update(sentence["tokens"][:max_len])
    words = sorted(word for word, count in vocab_counter.items() if count >= 2)
    vocab = {"<pad>": 0, "<unk>": 1, "<start>": 2, "<end>": 3}
    vocab.update({word: index + 4 for index, word in enumerate(words)})
    (output_dir / "vocab.json").write_text(json.dumps(vocab, ensure_ascii=False))

    summary = {}
    for split, items in selected_by_split.items():
        image_paths, encoded_captions = [], []
        for item in items:
            image_path = source_dir / "images" / item["filename"]
            if not image_path.is_file():
                raise FileNotFoundError(image_path)
            image_paths.append(str(image_path))
            sentences = item["sentences"][:CAPTIONS_PER_IMAGE]
            if len(sentences) < CAPTIONS_PER_IMAGE:
                sentences = sentences + [sentences[-1]] * (CAPTIONS_PER_IMAGE - len(sentences))
            for sentence in sentences:
                tokens = sentence["tokens"][:max_len]
                encoded = [vocab["<start>"]]
                encoded += [vocab.get(token, vocab["<unk>"]) for token in tokens]
                encoded += [vocab["<end>"]]
                encoded_captions.append(encoded)
        payload = {"IMAGES": image_paths, "CAPTIONS": encoded_captions}
        (output_dir / f"{split}_data.json").write_text(json.dumps(payload))
        summary[split] = (len(image_paths), len(encoded_captions))
    return vocab, summary


vocab, split_summary = prepare_cpu_quick_data(
    SOURCE_DIR, OUTPUT_DIR, SPLIT_LIMITS, MAX_LEN
)
print("CPU quick split:", split_summary, "vocab:", len(vocab))

## 3. 定义 Dataset 与批量读取

自定义数据集继承 `torch.utils.data.Dataset`，实现 `__len__` 和 `__getitem__`。`ImageTextDataset` 以 caption 为样本单位；每张图片连续对应5条 caption，因此 caption 下标整除5得到图片下标。

图像转换为 RGB，缩放到64×64，再转换为 `[3, 64, 64]` 浮点 tensor。caption 用 `<pad>` 补齐至22个位置，同时返回包含开始与结束标记的真实长度。普通 batch 输出图片、caption 与长度。

`make_loaders` 为训练、验证和测试分别构造 DataLoader；训练打乱顺序，评估保持顺序。`UniqueImageCaptionDataset` 额外提供每图仅一条描述的采样方式。

**任务与提示：** TODO 2 需要完成 caption 到图片的索引映射；不要把 caption 数误认为唯一图片数。

In [ ]:
IMG_SIZE = 224

train_transform = tv_transforms.Compose([
    tv_transforms.Resize(256),
    tv_transforms.RandomCrop(IMG_SIZE),
    tv_transforms.ToTensor(),
    tv_transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_transform = tv_transforms.Compose([
    tv_transforms.Resize(256),
    tv_transforms.CenterCrop(IMG_SIZE),
    tv_transforms.ToTensor(),
    tv_transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


def image_to_tensor(path, transform=None):
    image = Image.open(path).convert("RGB")
    if transform is not None:
        return transform(image)
    return val_transform(image)


class ImageTextDataset(Dataset):
    def __init__(self, data_path, vocab_path, captions_per_image=5, max_len=20, transform=None):
        self.data = json.loads(Path(data_path).read_text())
        self.vocab = json.loads(Path(vocab_path).read_text())
        self.cpi = captions_per_image
        self.max_len = max_len
        self.transform = transform

    def __len__(self):
        return len(self.data["CAPTIONS"])

    def __getitem__(self, index):
        # STUDENT_TODO 2：由 caption 下标映射到 image 下标
        # 提示：每张图片对应 self.cpi 条 caption，caption 下标整除 self.cpi 得到图片下标
        raise NotImplementedError("请完成 caption-image 映射")
        image = image_to_tensor(self.data["IMAGES"][image_index], self.transform)
        tokens = self.data["CAPTIONS"][index]
        length = len(tokens)
        padded = tokens + [self.vocab["<pad>"]] * (self.max_len + 2 - length)
        return image, torch.tensor(padded, dtype=torch.long), length


class UniqueImageCaptionDataset(ImageTextDataset):
    """检索训练集：每张图片只取一条 caption，避免同图 caption 成为假负例。"""
    def __len__(self):
        return len(self.data["IMAGES"])

    def __getitem__(self, image_index):
        caption_index = image_index * self.cpi + image_index % self.cpi
        image, caption, length = super().__getitem__(caption_index)
        return image, caption, length, self.data["IMAGES"][image_index]


def make_loaders(output_dir, batch_size=32, unique_train_images=False):
    output_dir = Path(output_dir)
    vocab_path = output_dir / "vocab.json"
    generator = torch.Generator().manual_seed(SEED)
    train_ds = ImageTextDataset(
        output_dir / "train_data.json", vocab_path,
        CAPTIONS_PER_IMAGE, MAX_LEN, transform=train_transform,
    )
    val_ds = ImageTextDataset(
        output_dir / "val_data.json", vocab_path,
        CAPTIONS_PER_IMAGE, MAX_LEN, transform=val_transform,
    )
    test_ds = ImageTextDataset(
        output_dir / "test_data.json", vocab_path,
        CAPTIONS_PER_IMAGE, MAX_LEN, transform=val_transform,
    )
    train_dataset = train_ds
    if unique_train_images:
        train_dataset = UniqueImageCaptionDataset(
            output_dir / "train_data.json", vocab_path,
            CAPTIONS_PER_IMAGE, MAX_LEN, transform=train_transform,
        )
    return (
        DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                   num_workers=0, generator=generator),
        DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0),
        DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0),
    )

<cell_type>markdown</cell_type>## 4. 图像编码器

使用 torchvision 提供的 ResNet-101 预训练模型作为图像编码器，去掉最后的平均池化层和全连接层，保留卷积特征图。输出形状为 `[B, 2048, 7, 7]`，即 49 个空间区域，每个区域 2048 维特征。

原作业模板使用随机初始化的 TinyCNN（3 层卷积，128 维，64×64 输入），视觉特征质量极低，导致训练效果很差。这里改用 ImageNet 预训练的 ResNet-101，与原始 ARCTIC 论文和参考代码 `caption.ipynb` 保持一致。

`grid=True` 返回 `[B, 2048, 7, 7]` 的局部网格特征；`grid=False` 返回 `[B, 2048]` 的全局向量。

**任务与提示：** TODO 3 完成卷积提取、池化与分支返回。

In [ ]:
class ImageEncoder(nn.Module):
    """使用预训练 ResNet-101 提取图像网格特征。"""
    def __init__(self, output_dim=2048, grid=True, finetuned=True):
        super().__init__()
        self.grid = grid
        # 提示：使用 tv_models.resnet101 加载预训练模型
        # 去掉最后两层（AvgPool 和 FC），只保留卷积部分作为 self.features
        # 可参考：nn.Sequential(*(list(resnet.children())[:-2]))
        resnet = tv_models.resnet101(weights=tv_models.ResNet101_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*(list(resnet.children())[:-2]))
        for param in self.features.parameters():
            param.requires_grad = finetuned
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, images):
        # STUDENT_TODO 3：提取卷积特征并按 grid 参数返回网格或全局特征
        # 提示：
        #   1. 用 self.features(images) 提取特征图，形状为 [B, 2048, 7, 7]
        #   2. 若 self.grid 为 True，直接返回特征图（供注意力机制使用）
        #   3. 若 self.grid 为 False，用 self.global_pool 池化后 flatten 为 [B, 2048]
        raise NotImplementedError("请完成图像编码器前向传播")


TinyImageEncoder = ImageEncoder

## 5. Additive Attention、文本解码器与 ARCTIC

ARCTIC 风格模型将图像局部特征与循环文本解码器结合。在每个生成时刻，模型根据当前 hidden state 选择相关视觉区域，再预测下一个词。

### 加性注意力

查询为 hidden state，键和值为16个图像区域向量：

$$e_j=w^T\tanh(W_q q+W_k k_j),\quad
\alpha_j=\mathrm{softmax}_j(e_j),\quad c=\sum_j\alpha_j k_j.$$

分别映射 query 和 keys，相加后经过 tanh 和评分层，沿区域维度归一化，再加权求和得到 context。TODO 4 完成打分表达式。

### 训练解码流程

1. 将网格展开为 `[B, 16, 128]`，由区域均值初始化 hidden state。
2. 取人工 caption 的当前词 embedding，采用 teacher forcing。
3. 计算 attention context，与词向量拼接后输入 GRUCell。
4. 分类层输出词表上的 logits，逐步收集预测值与 attention 权重。

默认词向量64维、hidden state 96维、attention 中间层64维。当前实现使用完整 batch 循环至最长有效序列，不进行动态 batch 缩减；损失阶段通过 mask 排除 padding。

### 生成与模型组装

`ARCTIC.forward` 连接 TinyCNN 与 AttentionDecoder。`greedy_decode` 从 `<start>` 开始，每一步选择 logits 最大的词，遇到 `<end>` 或长度上限停止。当前生成接口是贪心解码，不提供 beam_k 参数。

In [ ]:
class AdditiveAttention(nn.Module):
    def __init__(self, query_dim, key_dim, attention_dim):
        super().__init__()
        self.query_proj = nn.Linear(query_dim, attention_dim)
        self.key_proj = nn.Linear(key_dim, attention_dim)
        self.score = nn.Linear(attention_dim, 1)

    def forward(self, query, key_value):
        # STUDENT_TODO 4：实现 Additive Attention 的 energy 计算
        # 提示：
        #   1. 将 query 映射到 attention 空间并 unsqueeze(1) 以广播到所有 key
        #   2. 将 key_value 映射到 attention 空间
        #   3. 二者相加后经 tanh，再用 self.score 映射为标量，squeeze(-1) 得到 energy
        #   energy 形状应为 (batch_size, n_regions)
        raise NotImplementedError("请完成注意力打分")
        alpha = torch.softmax(energy, dim=1)
        context = (alpha.unsqueeze(-1) * key_value).sum(dim=1)
        return context, alpha


class AttentionDecoder(nn.Module):
    def __init__(self, image_dim, vocab_size, word_dim=512, hidden_dim=512, attention_dim=512):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, word_dim, padding_idx=0)
        self.attention = AdditiveAttention(hidden_dim, image_dim, attention_dim)
        self.init_hidden = nn.Linear(image_dim, hidden_dim)
        self.gru = nn.GRUCell(word_dim + image_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, vocab_size)

    def prepare_image(self, image_grid):
        return image_grid.flatten(2).transpose(1, 2)

    def forward(self, image_grid, captions, lengths):
        regions = self.prepare_image(image_grid)
        hidden = torch.tanh(self.init_hidden(regions.mean(dim=1)))
        embeddings = self.embedding(captions)
        steps = int(lengths.max().item()) - 1
        predictions, alphas = [], []
        for step in range(steps):
            context, alpha = self.attention(hidden, regions)
            hidden = self.gru(torch.cat([embeddings[:, step], context], dim=1), hidden)
            predictions.append(self.classifier(hidden))
            alphas.append(alpha)
        return torch.stack(predictions, dim=1), torch.stack(alphas, dim=1)

    def greedy_decode(self, image_grid, start_id, end_id, max_len=20):
        regions = self.prepare_image(image_grid)
        hidden = torch.tanh(self.init_hidden(regions.mean(dim=1)))
        current = torch.full((regions.size(0),), start_id, dtype=torch.long, device=regions.device)
        sequences = [[] for _ in range(regions.size(0))]
        finished = torch.zeros(regions.size(0), dtype=torch.bool, device=regions.device)
        for _ in range(max_len):
            context, _ = self.attention(hidden, regions)
            hidden = self.gru(torch.cat([self.embedding(current), context], dim=1), hidden)
            current = self.classifier(hidden).argmax(dim=1)
            for i, token in enumerate(current.tolist()):
                if not finished[i]:
                    sequences[i].append(token)
            finished |= current.eq(end_id)
            if finished.all():
                break
        return sequences


class ARCTIC(nn.Module):
    def __init__(self, vocab_size, image_dim=2048):
        super().__init__()
        self.encoder = ImageEncoder(image_dim, grid=True)
        self.decoder = AttentionDecoder(image_dim, vocab_size)

    def forward(self, images, captions, lengths):
        return self.decoder(self.encoder(images), captions, lengths)

<cell_type>markdown</cell_type>## 6. 损失函数、训练参数与评估

### 损失与优化

预测第 t+1 个词时，目标是 caption 中向后偏移一位的 token。`caption_loss` 按 `lengths - 1` 生成 mask，只对有效目标计算交叉熵（TODO 5）。`attention_coverage_loss` 同样排除 padding 时间步，再计算每个区域累计 attention 与1的平方差。

总损失为交叉熵加 `0.05 × attention_coverage_loss`。使用 Adam，梯度范数裁剪阈值为5。

### 参数设置

以下代码单元靠后位置的配置是当前实际使用的参数：

```python
config = SimpleNamespace(batch_size=32, epochs=10, learning_rate=5e-4)
```

即 batch size 为 32，共训练 **10 轮**，学习率为 **0.0005**。相比默认配置（batch_size=2, epochs=1, lr=1e-3），增大了训练轮数和 batch size，降低了学习率，以获得更稳定的训练和更好的效果。

### 模型改进说明

将图像编码器从随机初始化的 TinyCNN（128 维，64×64 输入）替换为 ImageNet 预训练的 ResNet-101（2048 维，224×224 输入），解码器隐藏层维度相应从 96 提升至 512，与原始 ARCTIC 论文保持一致。

### 流程与 BLEU-4

读取 batch → 前向传播（TODO 6）→ 损失与反向传播 → 梯度裁剪 → Adam 更新。每轮结束计算验证集 BLEU-4，并保存验证指标最好的参数副本。

评估对每张图片生成一条描述（TODO 7），对应5条人工 reference。去掉开始、结束和 padding 标记后，`corpus_bleu4` 汇总全语料1至4阶 clipped n-gram counts，结合长度惩罚计算分数，未使用平滑。任一阶无匹配时分数为0。

In [ ]:
def caption_loss(predictions, captions, lengths):
    steps = predictions.size(1)
    targets = captions[:, 1:steps + 1]
    mask = torch.arange(steps).unsqueeze(0) < (lengths - 1).unsqueeze(1)
    # STUDENT_TODO 5：只在有效 token 上计算交叉熵
    # 提示：用 mask 对 predictions 和 targets 进行布尔索引，
    # 然后调用 F.cross_entropy(predictions[mask], targets[mask])
    raise NotImplementedError("请完成 caption loss")


def attention_coverage_loss(alphas, lengths):
    steps = alphas.size(1)
    valid_steps = (
        torch.arange(steps, device=alphas.device).unsqueeze(0)
        < (lengths - 1).unsqueeze(1)
    )
    coverage = (alphas * valid_steps.unsqueeze(-1)).sum(dim=1)
    return ((1.0 - coverage) ** 2).mean()


def train_caption_epoch(loader, model, optimizer):
    model.train()
    total = 0.0
    for images, captions, lengths in loader:
        images, captions, lengths = images.to(device), captions.to(device), lengths.to(device)
        optimizer.zero_grad()
        # STUDENT_TODO 6：完成模型前向传播
        # 提示：调用 model(images, captions, lengths)，返回值为 (predictions, alphas)
        raise NotImplementedError("请完成训练前向传播")
        loss = caption_loss(predictions, captions, lengths)
        loss = loss + 0.05 * attention_coverage_loss(alphas, lengths)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total += loss.item() * images.size(0)
    return total / len(loader.dataset)


def corpus_bleu4(all_references, hypotheses):
    """标准 corpus BLEU-4：在整个语料上聚合 clipped n-gram counts。"""
    clipped = [0, 0, 0, 0]
    totals = [0, 0, 0, 0]
    hypothesis_length = 0
    reference_length = 0
    for references, hypothesis in zip(all_references, hypotheses):
        hypothesis_length += len(hypothesis)
        reference_length += min(
            (len(ref) for ref in references),
            key=lambda length: (abs(length - len(hypothesis)), length),
        )
        for n in range(1, 5):
            hyp_counts = Counter(
                tuple(hypothesis[i:i+n])
                for i in range(max(0, len(hypothesis) - n + 1))
            )
            max_ref_counts = Counter()
            for reference in references:
                ref_counts = Counter(
                    tuple(reference[i:i+n])
                    for i in range(max(0, len(reference) - n + 1))
                )
                for gram, count in ref_counts.items():
                    max_ref_counts[gram] = max(max_ref_counts[gram], count)
            clipped[n-1] += sum(
                min(count, max_ref_counts[gram]) for gram, count in hyp_counts.items()
            )
            totals[n-1] += sum(hyp_counts.values())
    if hypothesis_length == 0 or any(value == 0 for value in clipped):
        return 0.0
    precisions = [match / total for match, total in zip(clipped, totals)]
    brevity = min(1.0, math.exp(1.0 - reference_length / hypothesis_length))
    return brevity * math.exp(sum(math.log(p) for p in precisions) / 4.0)


def evaluate_caption(data_path, model, batch_size=16):
    payload = json.loads(Path(data_path).read_text())
    images = torch.stack([image_to_tensor(path) for path in payload["IMAGES"]])
    hypotheses = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(images), batch_size):
            grid = model.encoder(images[start:start+batch_size].to(device))
            # STUDENT_TODO 7：调用 greedy_decode 生成当前 batch 的 caption
            # 提示：调用 model.decoder.greedy_decode(grid, start_id, end_id, max_len)
            # 并用 hypotheses.extend(...) 收集结果
            raise NotImplementedError("请完成 caption 生成")
    all_references = []
    cleaned_hypotheses = []
    special = {vocab["<pad>"], vocab["<start>"], vocab["<end>"]}
    for i, hypothesis in enumerate(hypotheses):
        begin = i * CAPTIONS_PER_IMAGE
        refs = payload["CAPTIONS"][begin:begin + CAPTIONS_PER_IMAGE]
        all_references.append([[token for token in ref if token not in special] for ref in refs])
        cleaned_hypotheses.append([token for token in hypothesis if token not in special])
    return corpus_bleu4(all_references, cleaned_hypotheses)


config = SimpleNamespace(batch_size=32, epochs=10, learning_rate=5e-4)
train_loader, val_loader, test_loader = make_loaders(OUTPUT_DIR, config.batch_size)
model = ARCTIC(len(vocab)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
best_state, best_bleu = None, -1.0
for epoch in range(1, config.epochs + 1):
    loss = train_caption_epoch(train_loader, model, optimizer)
    val_bleu = evaluate_caption(OUTPUT_DIR / "val_data.json", model)
    print(f"epoch {epoch}/{config.epochs}: loss={loss:.4f}, val_BLEU-4={val_bleu:.4f}")
    if val_bleu >= best_bleu:
        best_bleu = val_bleu
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
model.load_state_dict(best_state)
test_bleu = evaluate_caption(OUTPUT_DIR / "test_data.json", model)
checkpoint_dir = NOTEBOOK_DIR / "work"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
torch.save({"model": best_state, "vocab": vocab, "test_bleu4": test_bleu}, checkpoint_dir / "arctic_resnet101.pt")
print(f"Test BLEU-4={test_bleu:.4f}")

## 理解与实验分析（20分）

请结合实际代码和运行结果回答。

### 1. 数据划分与词表（5分）

报告三个 split 的图片与 caption 数量，解释为何词表只从 train 构建，以及未登录词如何处理。

<cell_type>markdown</cell_type>**回答：**

请在此处作答。

### 2. 注意力与变长序列（5分）

解释16个区域的 attention 归一化维度，说明交叉熵与 coverage loss 为什么都需要长度 mask。

<cell_type>markdown</cell_type>**回答：**

请在此处作答。

### 3. 训练配置与生成（5分）

指出 epochs 与 learning_rate 的设置位置，解释 teacher forcing 与 greedy_decode 的输入区别。

<cell_type>markdown</cell_type>**回答：**

请在此处作答。

### 4. 结果与失败案例（5分）

填写两轮 loss、验证和测试 BLEU-4。可新增展示单元给出一张测试图片、生成描述及 reference，分析对象、属性或关系错误。

<cell_type>markdown</cell_type>**回答：**

请在此处作答。